<a href="https://colab.research.google.com/github/devwoo41/PromptEngineeringLecture/blob/master/13/RAG_exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q -U \
langchain langchain-core langchain-huggingface langchain-community \
langchain-chroma langchain-text-splitters \
huggingface_hub transformers accelerate \
chromadb sentence-transformers bs4

In [19]:
from huggingface_hub import notebook_login
import os
notebook_login()


In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 모델 이름 설정
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# 2. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          # GPU 있으면 자동으로 cuda 사용
    torch_dtype=torch.float16   # Colab GPU에서는 float16 권장
)

# 4. 추론 모드로 전환
#model.eval()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [21]:
import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# transformers pipeline 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.9,
    top_p=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# LangChain용 LLM 래핑
llm = HuggingFacePipeline(pipeline=pipe)

In [22]:
print(llm.invoke("What is the FLEX?"))


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 FLEX is a type of flexible joint that allows for a wide range of motion, making it ideal for various applications, including sports, dance, and even everyday activities. FLEX joints are made from a combination of materials, such as polyurethane, polyethylene, and polypropylene, which provide flexibility and durability.

FLEX joints are designed to be lightweight, yet strong and resistant to wear and tear. They are often used in applications where a high degree of flexibility is required, such as in sports equipment, dance gear, and other products that need to move freely.

Some of the key benefits of FLEX joints include:

*


#### 번외 - Kybalion-1B-DPO 로 돌려보기

In [23]:
'''
!pip install -U transformers accelerate langchain-huggingface huggingface_hub

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# 1. 모델 이름
model_name = "devwoo/Kybalion-1B-DPO"

# 2. tokenizer 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)

# pad_token 없으면 eos_token으로 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. model 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

# 4. pipeline 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# 5. LangChain LLM 래핑
llm = HuggingFacePipeline(pipeline=pipe)

response = llm.invoke("What is the FLEX?")
print(response)'''

'\n!pip install -U transformers accelerate langchain-huggingface huggingface_hub\n\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, pipeline\nfrom langchain_huggingface import HuggingFacePipeline\n\n# 1. 모델 이름\nmodel_name = "devwoo/Kybalion-1B-DPO"\n\n# 2. tokenizer 로드\ntokenizer = AutoTokenizer.from_pretrained(model_name)\n\n# pad_token 없으면 eos_token으로 설정\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\n\n# 3. model 로드\nmodel = AutoModelForCausalLM.from_pretrained(\n    model_name,\n    device_map="auto",\n    torch_dtype=torch.float16\n)\n\nmodel.eval()\n\n# 4. pipeline 생성\npipe = pipeline(\n    "text-generation",\n    model=model,\n    tokenizer=tokenizer,\n    max_new_tokens=128,\n    temperature=0.7,\n    top_p=0.9,\n    do_sample=True,\n    pad_token_id=tokenizer.eos_token_id,\n    return_full_text=False\n)\n\n# 5. LangChain LLM 래핑\nllm = HuggingFacePipeline(pipeline=pipe)\n\nresponse = llm.invoke("What is the FLEX?")\np

In [24]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://en.wikipedia.org/wiki/Hankuk_University_of_Foreign_Studies"
loader = WebBaseLoader(url)

docs = loader.load()
print("문서 개수:", len(docs))
print("문서 길이:", len(docs[0].page_content))
print("---")
print(docs[0].page_content[3000:4000])

문서 개수: 1
문서 길이: 31685
---
greenWebsitewww.hufs.ac.krKorean nameHangul한국외국어대학교Hanja韓國外國語大學校RRHanguk oegugeo daehakgyoMRHan'guk oegugŏ taehakkyo

Hankuk University of Foreign Studies (abbreviated as HUFS; Korean: 한국외국어대학교) is a private research university in Seoul, South Korea. The university currently teaches 45 foreign languages. In addition, it contains studies in humanities, law, political science, social sciences, business, medical science, natural sciences, and engineering. Main Building of Hankuk University of Foreign Studies (HUFS)

History[edit]
In April 1954, HUFS was founded as a college for studying foreign languages in by Kim Heung-bae with its first students studying English, French, Chinese, German, Spanish and Russian. Polish President Bronisław Komorowski giving a lecture at Hankuk University, October 2013.
In 2012, U.S President Barack Obama, during his visit to Korea, spoke at Hankuk University in Seoul about global progress toward nuclear non-proliferation.[2]
Through

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print("청크 개수:", len(splits))
print("---")
print(splits[5])

청크 개수: 47
---
page_content='History[edit]
In April 1954, HUFS was founded as a college for studying foreign languages in by Kim Heung-bae with its first students studying English, French, Chinese, German, Spanish and Russian. Polish President Bronisław Komorowski giving a lecture at Hankuk University, October 2013.
In 2012, U.S President Barack Obama, during his visit to Korea, spoke at Hankuk University in Seoul about global progress toward nuclear non-proliferation.[2]
Throughout its history, the university has been visited by numerous foreign dignitaries, including Crown Princess Victoria of Sweden,[3] Joko Widodo of Indonesia,[4] Viktor Orban of Hungary,[5] Abdullah Gul of Turkey,[6] Tsakhiagiin Elbegdorj of Mongolia,[7] Bronislaw Komorowski of Poland,[8] and many others.' metadata={'source': 'https://en.wikipedia.org/wiki/Hankuk_University_of_Foreign_Studies', 'title': 'Hankuk University of Foreign Studies - Wikipedia', 'language': 'en'}


In [26]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings()

vectorstore = Chroma.from_documents(
    documents = splits,
    embedding = embeddings,
)

print("Indexing 완료!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Indexing 완료!


In [27]:
retrieved = vectorstore.similarity_search("What is the Flex?")
print("검색된 청크 수:", len(retrieved))
print("---")
print(retrieved[0].page_content)

검색된 청크 수: 4
---
FLEX Center: The center was established to manage and operate FLEX (Foreign Language Examination). FLEX was developed by HUFS and is jointly administered with the Korea Chamber of Commerce and Industry. Tests are administered in seven languages: English, French, German, Russian, Spanish, Chinese, and Japanese.
HUFS TESOL Professional Education Center: In response to the rising demand for English teachers equipped with language and teaching knowledge, the HUFS TESOL (Teaching English to Speakers of Other Languages) Professional Education Center was established as an independent educational institution. The center offers all instruction required of English education professionals including teaching methods and application of theory to the field, as well as the evaluation, review, and production of learning materials in addition to English language training.


In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Prompt
template = '''Answer the question based only on the following context:{context}
Question: {question}'''
prompt = ChatPromptTemplate.from_template(template)

# Retriever
retriever = vectorstore.as_retriever()

# Combine Documents
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
rag_chain = (
  {"context": retriever | format_docs, "question": RunnablePassthrough()}
  | prompt
  | llm
  | StrOutputParser()
)

# Run Chain
response = rag_chain.invoke("What is the FLEX?")
print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 FLEX stands for Foreign Language Examination. It is a test that is administered by the Korea Chamber of Commerce and Industry and is used to evaluate the proficiency of foreign language speakers in seven languages: English, French, German, Russian, Spanish, Chinese, and Japanese.
